# Part 4: Evaluating Your RAG System

This guide provides instructions for the next critical step: quantitatively evaluating the performance of your RAG application. This will let you measure the quality of your chatbot's answers and understand its strengths and weaknesses.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

-----

## 1.&nbsp; Why We Need to Evaluate

You've built an amazing RAG chatbot. It *feels* like it works. But as engineers, we can't just rely on feelings. We need to *prove* it works and measure how well.

Answering the question "how good is it?" with just a few trial conversations is difficult. Evaluation provides a systematic way to measure performance using objective metrics. This notebook is about building a 'scorecard' to measure exactly how good our chatbot is.

A good evaluation framework lets us answer questions like:

**The Generation Stage(The LLM)**

* `Faithfulness`: Is the chatbot's answer factually based on the information in our documents, or is it making things up (hallucinating)?
* `AnswerCorrectness`: How well your response matches the meaning and facts of the ground truth.

**The Retrieval Stage(The Search)**
* `ContextPrecision`: Is the retrieval part of our system finding useful text chunks from our documents or lots of irrelevant chunks?
* `ContextRecall`: Is the retrieval part of our system finding all of the relevant pieces of info from our documents?

By the end of this guide, you'll have a repeatable process for testing your RAG system and generating a report of its performance.

-----
## 2.&nbsp; Create the Evaluation Directory

First, let's create a new home for our evaluation scripts.

1. In your VSCode file explorer, create a new folder named `evaluation` in the root of your project (at the same level as `src` and `data`).
2. Inside the new `evaluation` directory, create an empty `__init__.py` file to mark it as a Python package.

-----
## 3.&nbsp; A New Application

So far, we've worked entirely inside our `src` folder, building our chatbot application. Now, we're going to build a *second, separate* application in the `evaluation` folder.

You might be wondering, "Why not just add this to `src`?"

This is a crucial professional concept, just like the "Separation of Concerns" we learned in Part 1.

* Our **`src`** application is the "product." It's built to be run by a user.
* Our **`evaluation`** application is the "test lab." It's built to be run by *us* (the developers) to measure the quality of our product.

Keeping these separate is a best practice. Our test lab will need its own models, its own config, and its own logic, and we don't want to clutter our main application with it. Let's start building our test lab's components.

-----
## 4.&nbsp; Building the Evaluation Assets

Our evaluation requires two key assets: a set of questions to test the system and a configuration file for our evaluation settings.

### 4.1 Create the Evaluation Questions

To test our system, we need a consistent set of questions and corresponding "ground truth" answers. The ground truth is the ideal, factually correct answer we expect the RAG system to produce.

1.  In the `evaluation` directory, create a new file named `evaluation_questions.py`.
2.  Add the following list to this file. This defines our test dataset.

In [ ]:
EVALUATION_DATA: list[dict[str, str]] = [
    {
        "question": "Who is Alice and what is her sister doing at the "
                    "beginning of the story?",

        "ground_truth": "Alice is a young girl who goes on an adventure. "
                        "Her sister is sitting on the bank by her, reading "
                        "a book, but Alice finds it uninteresting because "
                        "it has no pictures or conversations."
    },
    # -----
    # It's best to test your code with only a question or two
    # until you're confident that it's working.
    # This saves a lot of your free API calls until you need them.
    # -----
    # {
    #     "question": "What does the White Rabbit take out of its "
    #                 "waistcoat-pocket?",
    #     "ground_truth": "The White Rabbit takes a watch out of its "
    #                     "waistcoat-pocket and looks at it before hurrying on."
    # },
    # {
    #     "question": "What is written on the bottle that Alice finds, and "
    #                 "what happens when she drinks from it?",
    #     "ground_truth": "The words 'DRINK ME' are printed on the bottle "
    #                     "in large letters. After drinking it, Alice "
    #                     "shrinks down to be only ten inches high."
    # },
    # {
    #     "question": "Describe the character of the Caterpillar.",
    #     "ground_truth": "The Caterpillar is a large, blue caterpillar found "
    #                     "sitting on top of a mushroom, smoking a long "
    #                     "hookah. It speaks in a sleepy, languid voice and "
    #                     "gives Alice short, sometimes cryptic advice."
    # },
    # {
    #     "question": "What is the capital of Germany?",
    #     "ground_truth": "The story of Alice's Adventures in Wonderland does "
    #                     "not contain information about the capital "
    #                     "of Germany."
    # },
    # {
    #     "question": "Why did the Mad Hatter and the March Hare put the "
    #                 "Dormouse in the teapot?",
    #     "ground_truth": "The story does not state a clear reason, but they "
    #                     "attempt to put the Dormouse in the teapot after "
    #                     "the Mad Hatter exclaims that the Dormouse is "
    #                     "asleep again and wants to change the subject of "
    #                     "conversation."
    # }
]


> Notice the question, "What is the capital of Germany?". This is an out-of-domain question. It tests whether our RAG system correctly uses the provided text and avoids making up answers when the information is not present.

### 4.2 Create the Evaluation Configuration

Next, we'll create a dedicated configuration file for our evaluation scripts.

1.  In the `evaluation` directory, create a new file named `evaluation_config.py`.
2.  Add the following code to the file. Here we define file paths for our results, specify which metrics we'll use, and set some sleep timers which we'll need later.

In [ ]:
from pathlib import Path

from ragas.metrics.base import Metric
from ragas.metrics import (
    Faithfulness,
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,

)

# --- LLM Model Configuration ---
EVALUATION_LLM_MODEL: str = "moonshotai/kimi-k2-instruct"

# --- Embedding Model Configuration ---
EVALUATION_EMBEDDING_MODEL_NAME: str = "BAAI/bge-large-en-v1.5"

# --- Paths for Evaluation ---
EVALUATION_ROOT_PATH: Path = Path(__file__).parent
EVALUATION_RESULTS_PATH: Path = EVALUATION_ROOT_PATH / "evaluation_results/"
EXPERIMENTAL_VECTOR_STORES_PATH: Path = (
    EVALUATION_ROOT_PATH
    / "evaluation_vector_stores/"
)
EVALUATION_EMBEDDING_CACHE_PATH: Path = (
    EVALUATION_ROOT_PATH
    / "evaluation_embedding_models/"
)

# --- Ragas Evaluation Metrics ---
EVALUATION_METRICS: list[Metric] = [
    Faithfulness(),
    AnswerCorrectness(),
    ContextPrecision(),
    ContextRecall(),
]

# --- Sleep Timers for API Limits ---
SLEEP_PER_EVALUATION: int = 60
SLEEP_PER_QUESTION: int = 6


-----
## 5.&nbsp; Building the Evaluation Components

Now we'll build the scripts for our new evaluation application. We'll use the same structure as our main chatbot: a 'parts store' for creating components (`evaluation_model_loader.py`), helper functions for specific jobs, and an 'engine' to orchestrate the process.

### 5.1 The Evaluation "Parts Store"

Just like our chatbot has `src/model_loader.py`, our evaluation lab needs one too. We need a separate one because we're using a different model for evaluation to take advantage of more API calls. It's also a good idea to use a different model for evaluation, as if you are using the same model, it's likely that it won't see it's own blindspots and can therefore, misleadingly, score itself higher.

1. In your VSCode editor, create a new file named `evaluation_model_loader.py` inside your `evaluation` folder.
2. Add the imports we'll need. This will look familiar, but notice we're importing our *new* evaluation config.

In [ ]:
import os
from dotenv import load_dotenv

from llama_index.llms.groq import Groq
from evaluation.evaluation_config import EVALUATION_LLM_MODEL
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.llms.base import LlamaIndexLLMWrapper

from evaluation.evaluation_config import (
    EVALUATION_EMBEDDING_MODEL_NAME,
    EVALUATION_EMBEDDING_CACHE_PATH,
)


3. Next, add the `initialise_evaluation_llm` function. Its job is simply to prepare the LLM we'll use for testing.

In [ ]:
# Load environment variables from the .env file
load_dotenv()


def initialise_evaluation_llm() -> Groq:
    """Initialises the Groq LLM with core parameters from config."""

    api_key: str | None = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError(
            "GROQ_API_KEY not found. "
            "Make sure it's set in your .env file."
        )

    return Groq(
        api_key=api_key,
        model=EVALUATION_LLM_MODEL,
    )

4. Finally, add the function to load the models for the `ragas` library. This function wraps our LlamaIndex LLM in a `LlamaIndexLLMWrapper` so `ragas` knows how to use it.

In [ ]:
def load_ragas_models(
) -> tuple[LlamaIndexLLMWrapper, HuggingFaceEmbeddings]:
    """
    Loads the LLM and embedding models
    required for Ragas evaluation.
    """
    print("--- 🧠 Loading Ragas LLM and Embeddings ---")

    llm_for_evaluation: Groq = initialise_evaluation_llm()

    # Wrap the LlamaIndex LLM for compatibility with Ragas
    ragas_llm = LlamaIndexLLMWrapper(llm=llm_for_evaluation)

    # Initialise the embedding model Ragas will use for its metrics
    ragas_embeddings = HuggingFaceEmbeddings(
        model=EVALUATION_EMBEDDING_MODEL_NAME,
        cache_folder=EVALUATION_EMBEDDING_CACHE_PATH.as_posix()
    )

    return ragas_llm, ragas_embeddings

### 5.2 Building the Helper Functions

Now let's create a file for our helper functions. These will handle specific, repeatable tasks like loading data, building an index, and saving results. We'll build it up function by function.

1. Create a new file named `evaluation_helper_functions.py` in the `evaluation` directory.
2. Add the necessary imports to the top of the empty file.

In [ ]:
from datasets import Dataset
from datetime import datetime
from pathlib import Path
import time
from typing import Any

from llama_index.core import (
    Document,
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import BaseQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import pandas as pd
from ragas.dataset_schema import EvaluationResult
from ragas.executor import Executor
from ragas.embeddings import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.llms.base import LlamaIndexLLMWrapper

from evaluation.evaluation_config import (
    EVALUATION_RESULTS_PATH,
    EXPERIMENTAL_VECTOR_STORES_PATH,
    SLEEP_PER_QUESTION,
    SLEEP_PER_EVALUATION,
    EVALUATION_METRICS,
)
from evaluation.evaluation_questions import EVALUATION_DATA
from src.config import DATA_PATH

3. Add our first helper function. Its only job is to read the `EVALUATION_DATA` list we created earlier and unpack it into two separate lists: one for questions and one for ground truth answers.

In [ ]:
def get_evaluation_data() -> tuple[list[str], list[str]]:
    """
    Extracts questions and ground truths from the EVALUATION_DATA constant.
    """

    return [item["question"] for item in EVALUATION_DATA], [
        item["ground_truth"] for item in EVALUATION_DATA
    ]

4. Next, add a function to manage our vector stores for experiments. It's important to keep these separate from the main vector store our chatbot uses. To save time, it will first check if a store for a specific experiment already exists. If it does, it loads it; otherwise, it builds a new one.

In [ ]:
def get_or_build_index(
    chunk_size: int, chunk_overlap: int, embed_model: HuggingFaceEmbedding
) -> VectorStoreIndex:
    """
    Checks for a persisted vector store for this experiment.
    If it exists, it loads it. If not, it builds it, persists it,
    and returns it.
    """

    vector_store_id: str = f"vs_chunk_{chunk_size}_overlap_{chunk_overlap}"
    specific_vector_store_path: Path = (
        EXPERIMENTAL_VECTOR_STORES_PATH
        / vector_store_id
    )

    if specific_vector_store_path.exists():
        print(f"--- Loading existing index from: {vector_store_id} ---")
        storage_context: StorageContext = StorageContext.from_defaults(
            persist_dir=str(specific_vector_store_path)
        )
        index: VectorStoreIndex = load_index_from_storage(
            storage_context, embed_model=embed_model
        )
    else:
        print(f"--- Creating new index for: {vector_store_id} ---")
        documents: list[Document] = SimpleDirectoryReader(
            input_dir=DATA_PATH
        ).load_data()

        text_splitter: SentenceSplitter = SentenceSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )

        index: VectorStoreIndex = VectorStoreIndex.from_documents(
            documents, transformations=[text_splitter], embed_model=embed_model
        )

        index.storage_context.persist(
            persist_dir=str(specific_vector_store_path)
        )
        print(f"--- Saved new index to: {vector_store_id} ---")
    return index

5. The `ragas` library requires the evaluation data to be in a specific format: a Hugging Face `Dataset` object. This next function handles that process. It iterates through each of our questions, queries the RAG engine to get an answer and the retrieved source context, and then organises this information alongside our ground truths into the required `Dataset` structure.

In [ ]:
def generate_qa_dataset(
    query_engine: BaseQueryEngine,
    questions: list[str],
    ground_truths: list[str]
) -> Dataset:
    """
    Generates answers and contexts for a given query engine
    and returns a HuggingFace Dataset.
    """

    responses: list[str] = []
    contexts: list[list[str]] = []
    for question_index, question in enumerate(questions):
        print(
            "Fetching context and synthesising response for question "
            f"{question_index + 1}/{len(questions)}: '{question[:30]}...'"
        )
        response_object = query_engine.query(question)
        responses.append(str(response_object))
        contexts.append(
            [node.get_content() for node in response_object.source_nodes]
        )

        # If you are hitting API rate limits
        # You can slow down the rate with time.sleep
        #
        # if question_index + 1 < len(questions):
        #     print(
        #         f"Taking a {SLEEP_PER_QUESTION} second breather "
        #         "to keep the API happy 🐢"
        #     )
        #     time.sleep(SLEEP_PER_QUESTION)
        # else:
        #     continue

    response_data: dict[str, list[Any]] = {
        "question": questions,
        "answer": responses,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }

    return Dataset.from_dict(response_data)

6. Now add the function that runs the `ragas` evaluation. This version will try to evaluate all the questions at once.

In [ ]:
def evaluate_without_rate_limit(
    qa_dataset: Dataset,
    ragas_llm: LlamaIndexLLMWrapper,
    ragas_embeddings: HuggingFaceEmbeddings,
) -> pd.DataFrame:
    """
    Runs Ragas evaluation on the entire dataset at once.
    Ideal for local models or APIs without strict rate limits.
    """

    print("--- ⚡ Running evaluation without rate limiting... ---")

    # Use ragas evaluation on the dataset
    result: EvaluationResult | Executor = evaluate(
        dataset=qa_dataset,
        metrics=EVALUATION_METRICS,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=True,
    )

    # Convert the evaluation result to a pandas DataFrame
    results_df: pd.DataFrame = result.to_pandas()

    print("--- ✅ Evaluation complete! ---")

    return results_df

7. Finally, add a utility function to save our results. To make experiment tracking easier, this function saves the output to the `evaluation_results` directory with a timestamp in the filename. It generates two files: a detailed CSV with the score for every single question and a summary CSV that shows the average scores across all questions.

In [ ]:
def save_results(results_df: pd.DataFrame, filename_prefix: str) -> None:
    """Saves the evaluation results and summary to CSV files."""

    results_dir: Path = EVALUATION_RESULTS_PATH
    results_dir.mkdir(exist_ok=True, parents=True)
    timestamp: str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    detailed_path: Path = (
        results_dir
        / f"{filename_prefix}_detailed_{timestamp}.csv"
    )
    results_df.to_csv(detailed_path, index=False)
    print(f"--- 💾 Detailed results saved to {detailed_path} ---")

    summary_path: Path = (
        results_dir
        / f"{filename_prefix}_summary_{timestamp}.csv"
    )
    # In the save_results function:
    param_cols: list[str] = [
        col
        for col in [
            'chunk_size',
            'chunk_overlap',
            'retriever_k',
            'reranker_n',
            'use_hyde']
        if col in results_df.columns
    ]

    if param_cols:
        avg_scores: pd.DataFrame = results_df.groupby(param_cols).mean(
            numeric_only=True
        )
        avg_scores.to_csv(summary_path)
        print(f"--- 💾 Summary of average scores saved to {summary_path} ---")

### 5.3 Building the Evaluation Engine

Now we can create the main orchestrator for our evaluation. It will use our helper functions to get the data, build a query engine, generate answers, run the `ragas` evaluation, and save the results.

1. Create a new file named `evaluation_engine.py` in the `evaluation` directory.
2. Add the required imports to the top of the file.

In [ ]:
from datasets import Dataset

from llama_index.core.indices import VectorStoreIndex
from llama_index.core.query_engine import (
    BaseQueryEngine,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
import pandas as pd
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.llms.base import LlamaIndexLLMWrapper

from evaluation.evaluation_helper_functions import (
    generate_qa_dataset,
    get_evaluation_data,
    get_or_build_index,
    save_results,
    evaluate_without_rate_limit,
    #evaluate_with_rate_limit,
)
from evaluation.evaluation_model_loader import load_ragas_models
from src.config import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    SIMILARITY_TOP_K,
)
from src.model_loader import get_embedding_model, initialise_llm

3. Add the main `evaluate_baseline` function. This function brings everything together to test the default configuration of our chatbot.

In [ ]:
def evaluate_baseline() -> None:
    """
    Evaluates the RAG system using only the settings from config.py.
    """

    print("--- 🚀 Stage 1: Evaluating Baseline Configuration ---")

    llm_to_test: Groq = initialise_llm()

    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    questions: list[str]
    ground_truths: list[str]
    questions, ground_truths = get_evaluation_data()

    index: VectorStoreIndex = get_or_build_index(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embed_model=embed_model_to_test
    )

    query_engine: BaseQueryEngine = index.as_query_engine(
        similarity_top_k=SIMILARITY_TOP_K,
        llm=llm_to_test
    )

    qa_dataset: Dataset = generate_qa_dataset(
        query_engine,
        questions,
        ground_truths
    )

    print("--- Running Ragas evaluation for baseline... ---")

    ragas_llm: LlamaIndexLLMWrapper
    ragas_embeddings: HuggingFaceEmbeddings
    ragas_llm, ragas_embeddings = load_ragas_models()

    results_df: pd.DataFrame = evaluate_without_rate_limit(
        qa_dataset,
        ragas_llm,
        ragas_embeddings,
    )

    # Add Chunk Size and Chunk Overlap to DataFrame to help tracking
    results_df['chunk_size'] = CHUNK_SIZE
    results_df['chunk_overlap'] = CHUNK_OVERLAP

    save_results(results_df, "baseline_evaluation")

    print("--- ✅ Baseline Evaluation Complete ---")

### 5.4 The Main Entry Point

Finally, we need a file to run our new evaluation application. Just like `main.py` runs our chatbot, `evaluate.py` will run our test lab.

1. Create a new file named `evaluate.py` in the **root** of your project folder.
2. Add the following code. It simply imports and runs our main evaluation function.

In [ ]:
from evaluation.evaluation_engine import (
    evaluate_baseline,
)


if __name__ == "__main__":

    # Run Stage 1: Baseline Evaluation
    evaluate_baseline()


-----

## 6.&nbsp; Run the Evaluation (and see what breaks)

With all the files in place, you're ready to run the evaluation.

From your VSCode terminal, run the following command:

In [ ]:
python evaluate.py

### 6.1 Houston, We Have a Problem: API Rate Limits

If you ran the evaluation (especially with all the questions uncommented in `evaluation_questions.py`), you probably ran into an error. This is a very common, real-world problem called 'rate limiting'.

Most APIs, including the Groq API, have a limit on how many requests you can send in a short period. Our current script, using `ragas.evaluate()`, tries to run all the evaluations in parallel to be fast. The API sees this burst of activity as a potential threat and blocks some of the requests, causing our script to fail.

Let's fix this by refactoring our code to be more patient.

### 6.2 Refactoring for Rate Limits

To solve this, we must change our strategy. Instead of evaluating the whole dataset at once, we'll evaluate it one row at a time and add a `time.sleep()` pause in between, just like a human taking a breather.

1. Open `evaluation/evaluation_helper_functions.py`.
2. Add the following new function to the file. This function will loop through our dataset and evaluate each question individually, pausing between each one.

In [ ]:
def evaluate_with_rate_limit(
    qa_dataset: Dataset,
    ragas_llm: LlamaIndexLLMWrapper,
    ragas_embeddings: HuggingFaceEmbeddings,
) -> pd.DataFrame:
    """
    Runs Ragas evaluation row-by-row to accommodate API rate limits,
    pausing between each evaluation.
    """

    print("--- 🐢 Running evaluation with rate limiting... ---")
    number_of_questions: int = len(qa_dataset)

    partial_results_list: list[pd.DataFrame] = []
    row: dict[str, Any]
    for i, row in enumerate(qa_dataset):
        print(
            f"Evaluating response for question {i + 1}/{number_of_questions}: "
            f"'{row['question'][:50]}...'"
        )

        # Create a new dataset with only one row to pass to evaluate
        single_row_dataset: Dataset = Dataset.from_dict(
            {key: [value] for key, value in row.items()}
        )

        # Evaluate just the single row
        result: EvaluationResult | Executor = evaluate(
            dataset=single_row_dataset,
            metrics=EVALUATION_METRICS,
            llm=ragas_llm,
            embeddings=ragas_embeddings,
            raise_exceptions=True,
        )

        # Convert the result to pandas DataFrame
        # Append the DataFrame to partial_results_list
        partial_results_list.append(result.to_pandas())

        # Pause to respect API rate limits, but not after the last item
        if i + 1 < number_of_questions:
            print(
                f"Taking a {SLEEP_PER_EVALUATION} second breather "
                "to keep the API happy."
            )
            time.sleep(SLEEP_PER_EVALUATION)

    # Combine the results from each row back into a single DataFrame
    results_df: pd.DataFrame = pd.concat(
        partial_results_list,
        ignore_index=True
    )

    print("--- ✅ Evaluation complete! ---")

    return results_df

3. Now, open `evaluation/evaluation_engine.py`.
4. First, import our new function at the top of the file.

    ```python
    from evaluation.evaluation_helper_functions import (
        # ... all the other imports
        #evaluate_without_rate_limit, <--Comment out this line
        evaluate_with_rate_limit, # <-- Remove comment from this line
    )
    ```

5. Finally, in the `evaluate_baseline` function, change the evaluation call. Comment out the call to `evaluate_without_rate_limit` and replace it with our new, safer function.

    ```python
    # ... inside evaluate_baseline ...
    
    # --- If you don't have a Rate per Minute limit on your API ---
    # results_df: pd.DataFrame = evaluate_without_rate_limit(
    #     qa_dataset,
    #     ragas_llm,
    #     ragas_embeddings,
    # )

    # --- If you do have a Rate per Minute API limit ---
    results_df: pd.DataFrame = evaluate_with_rate_limit(
        qa_dataset,
        ragas_llm,
        ragas_embeddings,
    )
    
    # ... rest of the function ...
    ```

-----
## 7.&nbsp; Run the Evaluation (for real this time)

Now that we've fixed the rate limiting issue, let's run the evaluation again from the VSCode terminal:

In [ ]:
python evaluate.py

The script will now print its progress, pausing between each question. It will be slower, but it will complete successfully.

### 7.1 Interpret the Results

Once the script finishes, you'll find a new `evaluation_results` folder inside your `evaluation` directory. This folder contains two CSV files with timestamps in their names:

1.  **`baseline_evaluation_detailed_...csv`**: This file contains the scores for each individual question across all the metrics.
2.  **`baseline_evaluation_summary_...csv`**: This file shows the average score for each metric across all questions, giving you a high-level view of the system's performance.

Open the summary file. You'll see the average scores for our key metrics:

* **faithfulness**: Measures if the answer is factually consistent with the retrieved context. A low score here indicates hallucination.
* **context_precision**: Measures the signal-to-noise ratio of the retrieved context. A low score means many of the retrieved chunks were irrelevant.
* **context_recall**: Measures if all the necessary information to answer the question was retrieved. A low score means the retriever missed important context.

-----
## 8.&nbsp; Congratulations! You're Now a RAG Engineer

This is it. You have successfully built a complete, end-to-end RAG system. You didn't just build a chatbot; you built the *professional testing harness* to evaluate and improve it.

You can now quantitatively measure its **Faithfulness** (if it's hallucinating) and its **Context Precision/Recall** (if the retriever is working). This is the complete, professional workflow, and you've built it from the ground up.

-----

## 9.&nbsp; Challenges and Further Exploration 😀

You've now learned how to build a complete evaluation pipeline using our "Alice in Wonderland" example. The final and most important step is to apply these techniques to the custom RAG chatbot you've been building.

These challenges will guide you through creating an evaluation framework for your own project, establishing its baseline performance, and examining the results to find opportunities for improvement.

### 1. Build the Evaluation Framework for Your RAG Chatbot

It's time to create the "scorecard" for your project. This involves creating a custom "answer key" with ground truths for your specific document and adapting the evaluation script to test your pipeline.

**Your Mission:**

1.  **Write Your Ground Truths:** Go to `evaluation/evaluation_questions.py` and replace the Alice in Wonderland questions and answers with your own. Write 5-7 challenging questions about the document you used in your RAG pipeline. Most importantly, you must also write the detailed, factually correct `ground_truth` answers for each.
2.  **Run Your Baseline Evaluation:** Execute the script from your terminal (`python evaluate.py`). It'll now use your questions to test your RAG pipeline. When it's finished, open the `_summary.csv` file. What is the baseline score for your custom RAG system?

> **Before you start:** Don't forget to go back to `evaluation/evaluation_questions.py` and uncomment the rest of the questions. We only used two to build the pipeline quickly, but now it's time for a full-scale test!

### 2. Analysing Your Results in a Notebook

Aggregate scores are useful, but the most valuable insights come from analysing individual failures. For this, we'll use a familiar friend: a Notebook. It's time to put on your detective hat and investigate your own project's results.

**Your Mission:**

1.  **Set Up Your Analysis Environment:**

      * In your project's root directory (at the same level as `src` and `evaluation`), create a new folder named `notebooks`. This is the standard place to keep your exploratory analysis separate from your application code.
      * Inside the new `notebooks` folder, create a new Notebook file. Name it `01_baseline_analysis.ipynb`. This clear naming scheme, with a numerical prefix, helps keep your experiments organised chronologically.

2.  **Load and Explore Your Results:**

      * Open your new notebook. In the first cell, you'll need to load the detailed results CSV file you generated. You'll need to replace the timestamped part of the filename with the one your script created.

    ```python
    import pandas as pd

    # Make sure to replace the timestamp with the one from your actual results file
    results_df = pd.read_csv("../evaluation/evaluation_results/baseline_evaluation_detailed_2025-10-21_09-10-00.csv")

    results_df.head()
    ```

3.  **Structure Your Investigation:**

      * Use Markdown cells and headings (`##`, `###`) to structure your analysis. A well-organised notebook is a key professional skill, as it makes your work easy for others (and your future self) to understand. Your notebook should tell a clear story.

4.  **Find and Analyse a Failure:**

      * In your notebook, find the question with the lowest score for a specific metric (for example, the lowest `faithfulness` or `context_recall` score).
      * Create a new section in your notebook for that metric (for example, `## Investigating Low Faithfulness`).
      * In that section, write a clear analysis explaining why you think the score was low. Use the data from the DataFrame (`question`, `contexts`, `answer`, and `ground_truth`) to support your conclusion.

This process of structured error analysis in a clean, well-documented notebook is one of the most important skills for building reliable and high-performing AI systems.

You should use your analysis to adapt your prompt and pipeline.